# SNR Calculation with photomancy.postproc

This notebook demonstrates SNR calculation using the **Mawet et al. (2014)** method.

| Method | Description | Use Case |
|--------|-------------|----------|
| **Mawet SNR** | Aperture-based with small-sample correction | Detection claims |

`DetectionEstimator.matched_filter(fwhm)` is available for comparison with Gaussian filtering and annular noise estimation.

In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Circle

# SNR API imports
from photomancy.postproc import DetectionEstimator
from photomancy.postproc.core.geometry import (
    calculate_n_apertures,
    generate_aperture_coords,
)

## Test Setup

In [ ]:
# Create test image
size = 101
fwhm = 5.0
noise_level = 100.0
center = size // 2

rng = np.random.default_rng(0)
image = rng.normal(0, noise_level, (size, size))

# Add a planet
planet_sep = 25
planet_flux = 500.0
sigma = fwhm / 2.355

planet_y = center + planet_sep
planet_x = center

y, x = np.ogrid[:size, :size]
r2 = (y - planet_y) ** 2 + (x - planet_x) ** 2
image += planet_flux * np.exp(-r2 / (2 * sigma**2))

image_jax = jnp.array(image)

## SNR (Aperture-Based)

The Mawet estimator places discrete apertures around the annulus at the planet's separation. It uses the **standard deviation** of aperture fluxes as the noise estimate, with a **small-sample correction** for few apertures.

Reference: Mawet et al. (2014), ApJ, 792, 97

In [ ]:
# Calculate SNR
positions = jnp.array([[planet_y, planet_x]])
estimator = DetectionEstimator.mawet(fwhm)
snr_val = float(estimator(image_jax, positions).statistic[0])

print(f"SNR: {snr_val:.2f}")

In [ ]:
# Visualize Mawet aperture placement using the actual geometry function
fig, ax = plt.subplots(figsize=(8, 8))

vmax = np.percentile(image, 99)
ax.imshow(image, origin="lower", cmap="viridis", vmin=-200, vmax=vmax)

# Calculate planet angle (same as used in the actual SNR calculation)
planet_angle = np.arctan2(planet_y - center, planet_x - center)

# Calculate number of apertures
n_apertures = calculate_n_apertures(radius=planet_sep, fwhm=fwhm)

# Generate aperture coords using the ACTUAL library function
y_coords, x_coords, mask = generate_aperture_coords(
    center=(center, center),
    radius=planet_sep,
    planet_angle=planet_angle,
    n_apertures=n_apertures,
    fwhm=fwhm,
)

# Draw planet aperture in red
planet_circle = Circle(
    (planet_x, planet_y), fwhm / 2, fill=False, color="red", linewidth=3
)
ax.add_patch(planet_circle)

# Draw reference apertures using the actual computed coordinates
y_arr = np.array(y_coords)
x_arr = np.array(x_coords)
mask_arr = np.array(mask)

for i in range(len(mask_arr)):
    if mask_arr[i]:
        circle = Circle(
            (x_arr[i], y_arr[i]),
            fwhm / 2,
            fill=False,
            color="white",
            linewidth=1.5,
            alpha=0.8,
        )
        ax.add_patch(circle)

ax.scatter([center], [center], marker="+", s=200, color="white", linewidths=2)
ax.set_title(
    f"SNR = {snr_val:.1f}\n{n_apertures} ref apertures at r={planet_sep}px",
    fontsize=12,
    fontweight="bold",
)
ax.set_xlabel("X (pixels)")
ax.set_ylabel("Y (pixels)")

# Legend
legend_elements = [
    Line2D([0], [0], color="red", linewidth=3, label="Planet aperture"),
    Line2D(
        [0],
        [0],
        color="white",
        linewidth=1.5,
        label=f"{n_apertures} reference apertures",
    ),
]
ax.legend(
    handles=legend_elements, loc="upper right", facecolor="black", labelcolor="white"
)

plt.colorbar(ax.images[0], ax=ax, label="e-", shrink=0.8)
plt.tight_layout()
plt.show()

## SNR vs Flux

SNR should scale linearly with planet flux. We average over multiple noise realizations to show the expected trend with error bars indicating statistical uncertainty.

In [ ]:
# Measure SNR across different flux levels with multiple noise realizations
flux_values = np.array([100, 200, 300, 400, 500, 600])
n_realizations = 10

# Collect SNR samples for each flux
snr_samples = np.zeros((len(flux_values), n_realizations))

for i, flux in enumerate(flux_values):
    for j in range(n_realizations):
        rng = np.random.default_rng(j)  # Different noise for each realization
        test_image = rng.normal(0, noise_level, (size, size))
        test_image += flux * np.exp(-r2 / (2 * sigma**2))
        test_jax = jnp.array(test_image)
        snr_samples[i, j] = float(estimator(test_jax, positions).statistic[0])

# Calculate mean and std
snr_mean = snr_samples.mean(axis=1)
snr_std = snr_samples.std(axis=1)

# Plot with error bars
fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(
    flux_values,
    snr_mean,
    yerr=snr_std,
    fmt="o-",
    label="SNR (mean +/- 1 sigma)",
    color="tab:blue",
    linewidth=2,
    capsize=4,
    capthick=1.5,
)
ax.axhline(y=5, color="red", linestyle=":", alpha=0.7, label="5 sigma threshold")

ax.set_xlabel("Planet Flux (e-)", fontsize=11)
ax.set_ylabel("SNR", fontsize=11)
ax.set_title(
    f"SNR vs Planet Flux ({n_realizations} noise realizations)",
    fontsize=12,
    fontweight="bold",
)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Using Estimators for Pipelines

For high-performance iterative pipelines, construct the estimator once to reuse the aperture kernel.

In [ ]:
import time

# Create estimator once
estimator = DetectionEstimator.mawet(fwhm, fast=True)

# Warmup (JIT compilation)
_ = estimator(image_jax, positions).statistic.block_until_ready()

# Time repeated calls
t0 = time.time()
for _ in range(100):
    estimator(image_jax, positions).statistic.block_until_ready()
elapsed = (time.time() - t0) / 100 * 1000

print(f"SNR Estimator: {elapsed:.2f} ms/call")

## Summary

`DetectionEstimator.mawet` reports the t-statistic, false positive fraction, and degrees of freedom at each candidate position.

For iterative pipelines, use the estimator:
```python
estimator = DetectionEstimator.mawet(fwhm, fast=True)
snrs = estimator(image, positions).statistic
```